# Pandas DataFrame 기초와 심화

이 노트북에서는 Pandas의 핵심 기능을 두 단계로 학습합니다.

**Part 1 — iris 데이터셋 (기초)**
- DataFrame 구조 이해, 컬럼 추출, 인덱싱
- 불린 필터링, 정렬, 시리즈 연산
- 정규화와 표준화
- 논리연산으로 3클래스 분류

**Part 2 — titanic 데이터셋 (심화)**
- 결측치, 중복값, 이상치 탐지 및 처리
- groupby, pivot_table, value_counts
- 데이터 타입 변환
- Feature Engineering

**사용 데이터**: seaborn 내장 데이터셋 (`iris`, `titanic`)

**선행 학습**: `numpy_basics.ipynb`

## Part 1 — iris 기초

iris 데이터셋은 붓꽃 150개체의 꽃받침(sepal)과 꽃잎(petal) 치수를 측정한 데이터입니다.
4개의 수치형 변수와 1개의 범주형 변수(species: setosa, versicolor, virginica)로 구성됩니다.

머신러닝 분류(classification) 실습에 가장 널리 사용되는 기준 데이터셋입니다.

**1-1. 데이터 로드 및 기본 확인**

In [ ]:
iris = sns.load_dataset('iris')
print("shape:", iris.shape)
print()
print(iris.dtypes)

In [ ]:
iris.describe()

In [ ]:
iris.info()

In [ ]:
iris.head()

In [ ]:
iris.tail()

**1-2. 컬럼 추출**

단일 컬럼을 추출하면 **Series**, 복수 컬럼을 추출하면 **DataFrame**이 반환됩니다.

In [ ]:
# 단일 컬럼 추출 → Series
sepal_len = iris['sepal_length']
print(type(sepal_len))
sepal_len.head()

In [ ]:
# 복수 컬럼 추출 → DataFrame (리스트를 인수로 전달)
sepal_cols = iris[['sepal_length', 'sepal_width']]
print(type(sepal_cols))
sepal_cols.head()

**1-3. 인덱싱 — loc과 iloc**

| 접근자 | 기준 | 끝 포함 여부 |
|---|---|---|
| `loc` | 레이블(이름) | 끝 포함 |
| `iloc` | 정수 위치 (0부터) | 끝 미포함 |

In [ ]:
# loc: 레이블 기반 — 행 인덱스 0~3, sepal_length 컬럼
iris.loc[0:3, 'sepal_length']

In [ ]:
# loc: 여러 컬럼 선택
iris.loc[0:3, ['sepal_length', 'species']]

In [ ]:
# iloc: 정수 위치 기반 — 행 0~2(3 미포함), 열 0(sepal_length)
iris.iloc[0:3, 0]

In [ ]:
# iloc: 행 10~14, 모든 컬럼
iris.iloc[10:15, :]

**1-4. 불린 필터링**

조건식이 True인 행만 선택합니다.
복합 조건은 `&`(and), `|`(or)를 사용하며, 각 조건을 괄호로 묶어야 합니다.

In [ ]:
# 단일 조건: sepal_length > 7.0
iris[iris['sepal_length'] > 7.0]

In [ ]:
# 복합 조건: sepal_length > 6.0 이고 species == 'virginica'
iris[(iris['sepal_length'] > 6.0) & (iris['species'] == 'virginica')]

In [ ]:
# query 메서드: 문자열로 조건 표현
iris.query("sepal_length > 6.0 and species == 'virginica'")

**1-5. 정렬**

`sort_values()`로 하나 또는 여러 컬럼 기준으로 정렬합니다.

In [ ]:
# 단일 컬럼 내림차순
iris.sort_values('sepal_length', ascending=False).head()

In [ ]:
# 다중 컬럼 정렬: species 오름차순, sepal_length 내림차순
iris.sort_values(['species', 'sepal_length'], ascending=[True, False]).head(10)

**1-6. 시리즈 연산**

Series는 NumPy 배열과 마찬가지로 벡터화 연산을 지원합니다.
모든 행에 동일한 연산을 반복문 없이 적용할 수 있습니다.

In [ ]:
# 두 컬럼을 곱해 근사 꽃받침 면적 컬럼 생성
iris['sepal_area_approx'] = iris['sepal_length'] * iris['sepal_width']
iris[['sepal_length', 'sepal_width', 'sepal_area_approx']].head()

In [ ]:
# apply로 사용자 정의 함수 적용: cm → mm 변환
iris['petal_length_mm'] = iris['petal_length'].apply(lambda x: x * 10)
iris[['petal_length', 'petal_length_mm']].head()

**1-7. 정규화 (Min-Max Normalization)**

값을 0~1 범위로 변환합니다.

$$x_{norm} = \\frac{x - x_{min}}{x_{max} - x_{min}}$$

이상치에 민감하지만, 범위가 명확한 데이터에 적합합니다.

In [ ]:
col = iris['sepal_length']
iris['sepal_length_norm'] = (col - col.min()) / (col.max() - col.min())

# 결과 확인: min=0, max=1이어야 합니다
print("min:", iris['sepal_length_norm'].min())
print("max:", iris['sepal_length_norm'].max())
iris[['sepal_length', 'sepal_length_norm']].head()

**1-8. 표준화 (Z-score Standardization)**

평균 0, 표준편차 1인 분포로 변환합니다.

$$x_{std} = \\frac{x - \\mu}{\\sigma}$$

정규분포를 가정하는 모델(선형 회귀, SVM 등)에 적합합니다.

In [ ]:
col = iris['sepal_length']
iris['sepal_length_std'] = (col - col.mean()) / col.std()

# 결과 확인: mean ≈ 0, std ≈ 1이어야 합니다
print("mean:", round(iris['sepal_length_std'].mean(), 6))
print("std: ", round(iris['sepal_length_std'].std(), 6))
iris[['sepal_length', 'sepal_length_std']].head()

**1-9. 논리연산으로 3클래스 분류**

petal_length의 기준치를 사용하여
setosa / versicolor / virginica 세 종을 근사적으로 분류합니다.

| 클래스 | 기준 |
|---|---|
| setosa | petal_length < 2.5 |
| versicolor | 2.5 ≤ petal_length < 4.9 |
| virginica | petal_length ≥ 4.9 |

이는 규칙 기반 분류(rule-based classification)의 간단한 예입니다.
실제 머신러닝 모델은 이 기준을 데이터에서 자동으로 학습합니다.

In [ ]:
# np.where 중첩으로 3가지 조건 적용
iris['species_pred'] = np.where(
    iris['petal_length'] < 2.5, 'setosa',
    np.where(iris['petal_length'] < 4.9, 'versicolor', 'virginica')
)

iris[['petal_length', 'petal_width', 'species', 'species_pred']].head(10)

In [ ]:
# 실제 species와 예측 비교
comparison = pd.crosstab(iris['species'], iris['species_pred'])
print(comparison)

correct = (iris['species'] == iris['species_pred']).sum()
total = len(iris)
print(f"\n정확도: {correct}/{total} = {correct/total:.1%}")

### 문제: petal_width 기준으로 3클래스 분류해보세요

iris 데이터의 `petal_width` 컬럼 분포를 확인한 뒤,
적절한 기준치를 설정하여 setosa / versicolor / virginica를 분류하는
`species_pred_w` 컬럼을 만들어보세요.

힌트: groupby로 각 species의 petal_width 통계를 먼저 확인하세요.
np.where를 중첩하여 조건을 적용합니다.

In [ ]:
# 여기에 코드를 작성하세요

## Part 2 — titanic 심화

타이타닉 데이터셋은 1912년 타이타닉호 승객 891명의 정보를 담고 있습니다.
실제 데이터 분석 작업에서 반드시 마주치는 결측치, 이상치, 범주형 변수 처리를
실습할 수 있는 대표적인 데이터셋입니다.

**주요 컬럼**

| 컬럼 | 설명 |
|---|---|
| survived | 생존 여부 (0=사망, 1=생존) |
| pclass | 객실 등급 (1=1등석, 2=2등석, 3=3등석) |
| sex | 성별 |
| age | 나이 |
| sibsp | 함께 탑승한 형제/배우자 수 |
| parch | 함께 탑승한 부모/자녀 수 |
| fare | 운임 요금 |
| embark_town | 탑승 항구 |

**2-1. 데이터 로드**

In [ ]:
titanic = sns.load_dataset('titanic')
print("shape:", titanic.shape)
print()
titanic.head()

In [ ]:
print(titanic.columns.tolist())

**2-2. 결측치 탐지 및 처리**

실제 데이터에는 반드시 결측치가 존재합니다.
처리 방법은 결측 비율과 컬럼의 의미에 따라 결정합니다.

| 상황 | 전략 |
|---|---|
| 수치형, 결측 비율 낮음 | 중앙값 또는 평균으로 대체 |
| 범주형 | 최빈값 또는 'Unknown' 카테고리 추가 |
| 결측 비율 50% 이상 | 컬럼 삭제 검토 |

In [ ]:
# 컬럼별 결측치 수
print(titanic.isnull().sum())

In [ ]:
# 컬럼별 결측 비율 (%)
missing_ratio = titanic.isnull().mean() * 100
print(missing_ratio.sort_values(ascending=False).round(1))

In [ ]:
# age: 결측 비율 약 20% → 중앙값으로 대체
age_median = titanic['age'].median()
titanic['age'] = titanic['age'].fillna(age_median)
print(f"age 중앙값: {age_median}, 결측치 잔여: {titanic['age'].isnull().sum()}")

In [ ]:
# embark_town: 결측 2건 → 최빈값으로 대체
embark_mode = titanic['embark_town'].mode()[0]
titanic['embark_town'] = titanic['embark_town'].fillna(embark_mode)
print(f"embark_town 최빈값: {embark_mode}")

In [ ]:
# deck: 결측 비율 약 77% → 분석에 활용하기 어려우므로 삭제
print(f"deck 결측 비율: {titanic['deck'].isnull().mean():.1%}")
titanic = titanic.drop(columns=['deck'])

In [ ]:
# 처리 후 결측치 확인
print(titanic.isnull().sum())

**2-3. 중복값 탐지 및 제거**

In [ ]:
dup_count = titanic.duplicated().sum()
print(f"중복 행 수: {dup_count}")
titanic = titanic.drop_duplicates()
print(f"제거 후 shape: {titanic.shape}")

**2-4. 이상치 탐지 — IQR 방법**

IQR(Interquartile Range) = Q3(75분위) - Q1(25분위)

정상 범위: [Q1 − 1.5×IQR, Q3 + 1.5×IQR]

이 범위를 벗어난 값을 이상치로 판단합니다.
타이타닉의 `fare` 컬럼은 일부 고가 요금으로 인해 우측 꼬리가 깁니다.

In [ ]:
Q1 = titanic['fare'].quantile(0.25)
Q3 = titanic['fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"정상 범위: [{lower:.2f}, {upper:.2f}]")

In [ ]:
fare_outliers = titanic[(titanic['fare'] < lower) | (titanic['fare'] > upper)]
print(f"이상치 건수: {len(fare_outliers)}")
print(f"이상치 비율: {len(fare_outliers)/len(titanic):.1%}")
fare_outliers[['pclass', 'sex', 'fare']].head(10)

### 문제: age 컬럼의 이상치를 IQR 방법으로 탐지해보세요

fare 컬럼에 적용한 IQR 방법을 `age` 컬럼에도 직접 적용해보세요.
이상치 건수와 해당 행의 나이 분포를 확인하세요.

힌트: quantile(0.25)와 quantile(0.75)를 사용하여 Q1, Q3를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요

**2-5. 행/컬럼 삭제**

분석에 불필요하거나 중복 정보를 담은 컬럼을 제거합니다.

In [ ]:
# 불필요 컬럼 삭제: alive(survived의 문자열 버전), class(pclass와 중복) 등
# who는 2-12 Title 파생에 사용하므로 나중에 삭제합니다
cols_to_drop = ['alive', 'class', 'adult_male', 'alone']
titanic = titanic.drop(columns=cols_to_drop)
print("남은 컬럼:", titanic.columns.tolist())

**2-6. 컬럼 이름 바꾸기**

`rename()`으로 컬럼명을 더 명확한 이름으로 변경합니다.

In [ ]:
titanic = titanic.rename(columns={
    'sibsp': 'siblings_spouse',
    'parch': 'parents_children'
})
print(titanic.columns.tolist())
titanic.head(3)

**2-7. value_counts**

범주형 컬럼의 빈도를 내림차순으로 확인합니다.

In [ ]:
print("sex 분포:")
print(titanic['sex'].value_counts())
print()
print("pclass 분포:")
print(titanic['pclass'].value_counts().sort_index())

In [ ]:
print("embark_town 분포 (비율):")
print(titanic['embark_town'].value_counts(normalize=True).round(3))

**2-8. groupby / agg**

특정 컬럼 값으로 데이터를 묶은 뒤 집계합니다.
SQL의 `GROUP BY`와 동일한 개념입니다.

In [ ]:
# pclass별 생존률
print("pclass별 생존률:")
print(titanic.groupby('pclass')['survived'].mean().round(3))

In [ ]:
# sex별 평균 운임
print("sex별 평균 운임:")
print(titanic.groupby('sex')['fare'].mean().round(2))

In [ ]:
# pclass별 fare 다중 통계
titanic.groupby('pclass')['fare'].agg(['mean', 'median', 'std']).round(2)

In [ ]:
# pclass × sex 별 생존률
titanic.groupby(['pclass', 'sex'])['survived'].mean().round(3).unstack()

**2-9. pivot_table**

groupby 집계 결과를 행/열 교차표 형태로 표현합니다.

In [ ]:
pd.pivot_table(
    titanic,
    values='survived',
    index='pclass',
    columns='sex',
    aggfunc='mean'
).round(3)

### 문제: pivot_table로 embark_town × pclass 교차표를 만들어보세요

탑승 항구(embark_town)와 객실 등급(pclass)을 기준으로
평균 운임(fare)을 교차표로 나타내보세요.

힌트: pd.pivot_table의 index, columns, values, aggfunc 인수를 설정하세요.

In [ ]:
# 여기에 코드를 작성하세요

**2-10. 데이터 타입 변환**

`survived` 컬럼은 0/1 정수이지만, 의미상 참/거짓(Boolean)입니다.

In [ ]:
print("변환 전:", titanic['survived'].dtype)
titanic['survived'] = titanic['survived'].astype(bool)
print("변환 후:", titanic['survived'].dtype)
titanic[['survived']].head()

**2-11. Feature Engineering — 가족 수**

원본 데이터에 없는 새로운 변수를 만들어 분석에 활용합니다.

`siblings_spouse`(형제/배우자)와 `parents_children`(부모/자녀)을 합산하고
본인을 포함(+1)하여 `family_size`를 계산합니다.

In [ ]:
titanic['family_size'] = titanic['siblings_spouse'] + titanic['parents_children'] + 1
print(titanic['family_size'].value_counts().sort_index())

In [ ]:
# 가족 규모별 생존률
titanic.groupby('family_size')['survived'].mean().round(3)

**2-12. Feature Engineering — Title(경칭) 파생**

seaborn의 titanic 데이터셋에는 `who` 컬럼(man/woman/child)이 내장되어 있습니다.
이 컬럼과 `sex`, `age`를 조합하여 전통적인 경칭(Mr/Mrs/Miss/Master)을 파생시킵니다.

실제 이름 문자열이 있는 경우에는 정규표현식으로 경칭을 직접 추출합니다.

```python
# 이름 형식: "Braund, Mr. Owen Harris"
df['title'] = df['name'].str.extract(r',\\s*([^\\.]+)\\.')
```

여기서는 who/sex/age 규칙으로 동일한 결과를 재현합니다.

In [ ]:
# who(man/woman/child)와 sex, age로 전통 경칭 파생
def assign_title(row):
    if row['who'] == 'child':
        return 'Master' if row['sex'] == 'male' else 'Miss'
    elif row['sex'] == 'male':
        return 'Mr'
    else:
        # 결혼 여부를 age 기준으로 근사 (25세 이상 Mrs)
        return 'Mrs' if row['age'] >= 25 else 'Miss'

titanic['title'] = titanic.apply(assign_title, axis=1)
# who 컬럼은 title 파생 후 삭제
titanic = titanic.drop(columns=['who'])
print(titanic['title'].value_counts())

In [ ]:
# Title별 평균 나이
title_age_mean = titanic.groupby('title')['age'].mean().sort_values(ascending=False)
print(title_age_mean.round(1))

### 문제: title별 생존률을 구해보세요

titanic 데이터에서 title별 생존률(survived 평균)을 구하세요.
어떤 경칭을 가진 승객의 생존률이 높은지 확인해보세요.

힌트: groupby('title')['survived'].mean()을 사용하세요.
survived 컬럼은 bool 타입이므로 mean()을 적용하면 생존률(비율)이 계산됩니다.

In [ ]:
# 여기에 코드를 작성하세요